# Solutions · Chapter 04-02 · Baselines first, always

Worked answers for `notebooks/04_workflow/04-02_baselines.ipynb`.

Three of these (E9, E11, E20) have answers that contradict the obvious prediction. Those are the ones
worth reading even if you got the rest.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


# SYNTHETIC. The gym panel from 04-01, framed as in 04-02.
def load_members():
    rng = np.random.default_rng(41)
    n = 600
    join_month = rng.integers(1, 13, n)
    commitment = rng.beta(2.0, 2.0, n)
    rows = []
    for member in range(n):
        drifting = 0.0
        base_visits = 2 + 10 * commitment[member]
        for month in range(join_month[member], 25):
            drifting += rng.normal(0.0, 0.35)
            visits = max(0, int(round(rng.poisson(max(0.2, base_visits - drifting)))))
            tickets = int(rng.random() < 0.05 + 0.10 * (visits == 0))
            hazard = 1 / (1 + np.exp(3.0 + 2.5 * commitment[member] - 0.55 * max(0, 4 - visits)))
            cancelled = int(rng.random() < hazard)
            rows.append((member + 1, month, visits, tickets, cancelled))
            if cancelled:
                break
    return pd.DataFrame(rows, columns=["member_id", "month", "visits", "tickets", "cancelled"])


panel = load_members().sort_values(["member_id", "month"])
panel["next_visits"] = panel.groupby("member_id").visits.shift(-1)
table = panel.dropna(subset=["next_visits"]).copy()
table["previous_visits"] = table.groupby("member_id").visits.shift(1)
table["mean_so_far"] = (table.groupby("member_id").visits
                        .expanding().mean().reset_index(level=0, drop=True))
table = table.dropna(subset=["previous_visits"])

train = table[table.month <= 16]
test = table[table.month > 16]
actual = test.next_visits.to_numpy()
columns = ["visits", "previous_visits", "mean_so_far", "tickets", "month"]


def mean_absolute(prediction):
    return float(np.mean(np.abs(actual - prediction)))


BEST_BASELINE = mean_absolute(test.mean_so_far.to_numpy())
print("set up. best baseline (per-member running mean) MAE %.4f" % BEST_BASELINE)

## Quick understanding

### E1

**Regression:** global mean, global median, persistence (last value), per-entity mean.
**Classification:** majority class, base-rate guessing (AUC 0.5), best single-column rule.

What beating each proves:

| Baseline | Beating it proves |
|---|---|
| Global mean / median | the features carry *any* information about the target |
| Persistence | you know more than "nothing changes" |
| Per-entity mean | you have learned something beyond the identity of the entity |
| Majority class | your predictions are better than always guessing the common label |
| Base rate (AUC 0.5) | your ranking is better than shuffling |
| Best single rule | the model is worth more than one threshold on one column |

### E2

`skill = (baseline error - model error) / baseline error`.

- **0%** - the model matches the baseline exactly. Every hour spent on it bought nothing.
- **100%** - the model is perfect (error zero). In practice this means leakage.
- **-5%** - the model is 5% *worse* than the free rule. It should not ship.

### E3

Because **the fitted parameters carry sampling noise and the rule has no parameters to be noisy.**

The models were handed `mean_so_far` as a column, so in principle either could have set its coefficient
to 1, everything else to 0, and matched the baseline exactly. Neither did, because they were fitted to
minimise error on 4,830 training rows from months 1-16, and the coefficients that do that best are not
the coefficients that do best on months 17-23. The difference is overfitting, in its smallest and most
ordinary form - no dramatic memorisation, just a handful of coefficients tuned very slightly to the wrong
months.

A rule with no parameters cannot make that mistake. **It cannot get better with fitting and it cannot get
worse with it either**, which is exactly why it is the right thing to measure against.

## Hand calculation

### E4

- First model: `(40 - 34) / 40` = **15.00%** skill.
- Second: `(40 - 30) / 40` = **25.00%** skill.
- Improvement over the previous *model*: `(34 - 30) / 34` = **11.76%**.

**Put the skill against the baseline in the slide - 15% then 25%.** Both numbers are arithmetically
correct, and the 11.76% is the one that misleads, because it is measured against a moving reference.
Quote improvements against the previous model often enough and you can report a large total from a
sequence of tiny gains, while the honest total against the fixed baseline is 25%.

The rule that avoids the whole problem: **fix one baseline at the start of the project and report every
model against it, forever.** It is the only reference that does not move when you are the one moving it.

### E5

TP 12, FP 8, FN 28, TN 952.

- **(a)** Majority class: 960 of 1,000 are negative, so **96.0%**.
- **(b)** The rule: `(12 + 952) / 1000` = **96.4%**.
- **(c)** Precision `12/20` = **60%**, recall `12/40` = **30%**.

**The rule is far more useful, and the accuracies barely notice.** 96.4% against 96.0% - four tenths of a
percentage point - for a rule that finds 30% of all the fraud with a 60% hit rate, which is an
operationally excellent detector.

The accuracies disagree with the verdict because **accuracy is dominated by the 960 negatives**, which
both predictors get almost entirely right. All the value is in the 40 positives, and accuracy weights
them at 4% of the score. Precision and recall look only at where the action is, which is why they are
the right pair for a rare class (06-05).

In [ ]:
print("E4  skill of model 1: %.2f%%   model 2: %.2f%%   model 2 over model 1: %.2f%%"
      % (100 * (40 - 34) / 40, 100 * (40 - 30) / 40, 100 * (34 - 30) / 34))
print()
true_positive, false_positive = 12, 8
false_negative, true_negative = 40 - 12, 960 - 8
print("E5  majority-class accuracy %.1f%%" % (100 * 960 / 1000))
print("    rule accuracy          %.1f%%" % (100 * (true_positive + true_negative) / 1000))
print("    precision %.0f%%, recall %.0f%%"
      % (100 * true_positive / (true_positive + false_positive),
         100 * true_positive / (true_positive + false_negative)))

### E6

Both are describing the same forest. **Team A is quoting skill against the global median (2.9688), where
the forest is +17.97%. Team B is quoting it against the per-member running mean (2.3640), where the same
forest is -3.02%.** Neither has done any arithmetic wrong.

**Team B, without hesitation.** Not because their number is lower - because they computed the stronger
baseline. Team A's report is not false, it is incomplete in the one way that matters: it would lead to
deploying a random forest to do worse than a `groupby`.

This is the most common way results mislead in practice, and it is very rarely deliberate. Somebody
computes the obvious baseline, beats it comfortably, and never asks whether a better free rule exists.
**The defence is to make "what is the strongest baseline I can think of in ten minutes?" a required step,
not an optional one.**

### E7

Visits 6, 8, 5, 9, 3.

- **Persistence** predicts the last value: **3**.
- **Running mean** predicts `(6+8+5+9+3)/5` = **6.2**.

The member visits 4 times.

- Persistence error: `|4 - 3|` = **1.0**
- Running mean error: `|4 - 6.2|` = **2.2**

**Persistence wins this month, by more than a factor of two.** And it tells you **nothing**. This is a
sample of size one. The chapter's comparison used 3,049 test rows to conclude that the running mean beats
persistence comfortably - 2.3640 against 3.0482 - and a single month landing the other way is exactly what
an average advantage of that size looks like from close up.

The habit worth taking: **never adjudicate between two rules on one example**, however vivid. It is the
same mistake as reading a trend off one train/test split (04-01, E10), and it is the reason error bars
exist.

In [ ]:
history = [6, 8, 5, 9, 3]
observed = 4
print("E7  persistence predicts %d, error %.1f" % (history[-1], abs(observed - history[-1])))
print("    running mean predicts %.1f, error %.1f"
      % (np.mean(history), abs(observed - np.mean(history))))
print("    on this one month persistence wins. On the chapter's 3,049 rows it loses by %.1f%%."
      % (100 * (mean_absolute(test.visits.to_numpy()) - BEST_BASELINE) / BEST_BASELINE))

## Coding

### E8 - reporting helpers

In [ ]:
def skill(baseline_error, model_error):
    return (baseline_error - model_error) / baseline_error


def report(name, prediction, baseline_error=BEST_BASELINE, baseline_name="per-member mean"):
    error = mean_absolute(prediction)
    print("%-34s MAE %.4f   %+.2f%% vs %s"
          % (name, error, 100 * skill(baseline_error, error), baseline_name))
    return error


from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

linear = LinearRegression().fit(train[columns], train.next_visits)
forest = RandomForestRegressor(n_estimators=200, random_state=0, min_samples_leaf=5)
forest.fit(train[columns], train.next_visits)

report("global mean", np.full(len(test), train.next_visits.mean()))
report("global median", np.full(len(test), train.next_visits.median()))
report("persistence", test.visits.to_numpy())
report("per-member mean", test.mean_so_far.to_numpy())
report("linear regression", linear.predict(test[columns]))
report("random forest", forest.predict(test[columns]))

### E9 - does a shorter window beat the full running mean?

The intuition says yes: a member's habits change, so recent months should be more relevant than months a
year ago. The intuition is wrong here, and the shape of the answer is the interesting part.

In [ ]:
rows = []
for window in [1, 2, 3, 6, 12]:
    rolling = (table.groupby("member_id").visits
               .rolling(window, min_periods=1).mean().reset_index(level=0, drop=True))
    rows.append({"window (months)": window,
                 "MAE": round(mean_absolute(rolling[test.index].to_numpy()), 4)})
rows.append({"window (months)": "all", "MAE": round(BEST_BASELINE, 4)})
windows = pd.DataFrame(rows)
windows["skill vs full mean"] = ["%+.2f%%" % (100 * skill(BEST_BASELINE, m)) for m in windows.MAE]
print(windows.to_string(index=False))

**No. The longer the window, the better, all the way to using everything.** 2.5444 at three months
against 2.3640 for the full history, and the improvement is still going at twelve months (2.3689).

The trade-off in the question is real - more data against more relevant data - and on this dataset the
first term wins outright, for the same reason persistence lost. Each month is a noisy count around a
member's underlying rate, and that rate drifts only slowly. Averaging twelve noisy months of a
slowly-drifting quantity beats averaging three, because the noise you remove is larger than the drift you
smear over.

**Which term wins is an empirical question about your data, not a principle**, and it is answered by the
five lines above. On data where behaviour changes fast - a subscription after a price rise, sales after a
product launch - the same table would slope the other way, and a short window would win. Compute it; do
not reason about it.

### E10 - give the forest a fairer chance

In [ ]:
rows = []
for subset in [["visits", "previous_visits", "mean_so_far", "tickets", "month"],
               ["mean_so_far", "visits"],
               ["mean_so_far"]]:
    tree_model = RandomForestRegressor(n_estimators=200, random_state=0, min_samples_leaf=5)
    tree_model.fit(train[subset], train.next_visits)
    line_model = LinearRegression().fit(train[subset], train.next_visits)
    rows.append({"columns": ", ".join(subset),
                 "forest MAE": round(mean_absolute(tree_model.predict(test[subset])), 4),
                 "linear MAE": round(mean_absolute(line_model.predict(test[subset])), 4)})
print(pd.DataFrame(rows).to_string(index=False))
print()
print("the baseline: %.4f" % BEST_BASELINE)

**Removing columns helps the linear model slightly and hurts the forest**, and neither reaches the
baseline.

Linear goes 2.3992 -> 2.3921 with two columns: dropping `month` and `tickets` removes two coefficients
that were fitting noise, which is what the "fewer features" instinct predicts.

The forest goes 2.4353 -> 2.4766 -> 2.4864, getting **worse** as columns are removed. That looks
backwards until you remember what a tree does: it can only predict by splitting, so with a single
continuous column it approximates a smooth relationship by a staircase, and the extra columns were giving
it somewhere to put the splits. **A forest handed exactly one useful feature is a worse version of a
lookup table.**

So the instinct is half right, and the useful correction is that **"more features is better" and "fewer
features is better" are both wrong as rules** - the question is whether each column carries signal for
*this* model class, and the answer differs by model. What is not in doubt is the conclusion: 2.3921 at
best against a baseline of 2.3640. The gap survives every variation tried in this chapter.

### E11 - the per-member median

03-01 says the median minimises absolute error, and the metric here is absolute error. So the running
median should win. **Predict, then run.**

In [ ]:
running_median = (table.groupby("member_id").visits
                  .expanding().median().reset_index(level=0, drop=True))
report("per-member running median", running_median[test.index].to_numpy())
report("per-member running mean", test.mean_so_far.to_numpy())
print()
sample = table[table.member_id == test.member_id.value_counts().index[0]]
print("why: the median of a short run of small counts is coarse")
print("  a member's visits :", sample.visits.head(8).to_list())
print("  running median    :", running_median[sample.index[:8]].round(2).to_list())
print("  running mean      :", sample.mean_so_far.head(8).round(2).to_list())

**The mean wins: 2.3640 against 2.4024.** 03-01 is not wrong, and the reason the prediction fails is
worth the exercise.

03-01's result is that the median minimises the absolute error **of a fixed set of numbers you already
have**. That is not this problem. Here the summary is an *estimate of a member's underlying rate*, used
to predict a **future** draw, and the two tasks have different best answers:

- For predicting a future draw under absolute error, the ideal prediction is the median of the
  **predictive distribution** - and for a count fluctuating around a rate of 7, that is about 7, which is
  what the mean estimates.
- The **mean is the more efficient estimator of that rate**: it uses every observation, while the median
  effectively uses the middle one or two. With only thirteen months of history that difference is
  substantial.
- And the running median of small integers is **coarse**, moving in steps and often landing on a whole or
  half number, as the printout shows.

**The general lesson is the one that keeps coming back: a result about summarising the data you have is
not automatically a result about predicting data you do not have.** The two coincide often enough to be
misleading. Checking costs one line.

### E12 - the best single rule on each column

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

CUT, HORIZON = 12, 6
active = panel[(panel.month == CUT) & (panel.cancelled == 0)].member_id.unique()
history = panel[panel.member_id.isin(active) & (panel.month <= CUT)]
future = panel[panel.member_id.isin(active) & (panel.month > CUT)]
left = future[(future.month <= CUT + HORIZON) & (future.cancelled == 1)].member_id.unique()
churn = pd.Series(np.isin(active, left).astype(int), index=active)

features = pd.DataFrame({
    "visits_at_cut": history[history.month == CUT].set_index("member_id").visits.reindex(active),
    "mean_visits_last_3": history[history.month > CUT - 3].groupby("member_id").visits.mean().reindex(active),
    "tenure_months": history.groupby("member_id").size().reindex(active),
    "tickets_so_far": history.groupby("member_id").tickets.sum().reindex(active),
})
train_X, test_X, train_y, test_y = train_test_split(
    features, churn, test_size=0.3, random_state=0, stratify=churn)

rows = []
for column in features.columns:
    best_auc, best_threshold = max((roc_auc_score(train_y, (train_X[column] <= t).astype(int)), t)
                                   for t in np.unique(train_X[column]))
    rows.append({"column": column, "best threshold": round(float(best_threshold), 2),
                 "AUC on train": round(best_auc, 4),
                 "AUC on test": round(roc_auc_score(test_y, (test_X[column] <= best_threshold).astype(int)), 4)})

centre, spread = train_X.mean(), train_X.std()
model = LogisticRegression(max_iter=2000).fit((train_X - centre) / spread, train_y)
rows.append({"column": "logistic regression, all four", "best threshold": None,
             "AUC on train": round(roc_auc_score(train_y, model.predict_proba((train_X - centre) / spread)[:, 1]), 4),
             "AUC on test": round(roc_auc_score(test_y, model.predict_proba((test_X - centre) / spread)[:, 1]), 4)})
print(pd.DataFrame(rows).to_string(index=False, na_rep="-"))

**`mean_visits_last_3` wins, and the model beats it by 0.0048 of AUC** - 0.6359 against 0.6311. On a test
set of 157 members with 21 cancellers, that difference is not measurable.

The honest summary: **the entire model is worth approximately one threshold on one column**, and a gym
manager could implement it as "call anyone averaging under seven visits a month".

Two further things in that table.

`tenure_months` and `tickets_so_far` both return **exactly 0.5000**. Their best threshold is degenerate -
it flags everybody or nobody - which is what "this column contains no usable signal on its own" looks
like numerically.

And `mean_visits_last_3` scores **0.7585 on train, 0.6311 on test**. That drop of 0.13 is not the
column's fault; it is the cost of having chosen its threshold by looking at the training data. **Even a
one-parameter model overfits when you fit the parameter.** A baseline you tune is no longer free, and it
needs held-out evaluation exactly like anything else.

## Interpretation

### E13

**What you know immediately:** predicting "not fraud" for every transaction scores **99.5%**. Their model
is 0.3 percentage points *worse* than doing nothing, so on the number they have chosen to report it has
negative skill.

**The two numbers to ask for:** at a fixed alert budget - say the number of transactions a human team can
review per day - **how many frauds does it catch (recall), and what share of the alerts are real
(precision)?** Those are the only two that describe what the system would do in operation.

Said gently, because the colleague has almost certainly not realised: accuracy on a 0.5% base rate is a
measure of how often you say "no", and both of you already know the answer is "almost always".

### E14

**Recommend persistence, or a hybrid, and say why in revenue rather than in MAE.**

The 30% MAE improvement is an average over products, and MAE weights every product equally regardless of
volume. The ten largest products are 70% of revenue and the model is worse on all of them, so on the part
of the business that matters the model is a regression - and the aggregate number hides it completely
because hundreds of low-volume products are being forecast slightly better.

**The number to ask for: error weighted by volume, or the same comparison computed in currency rather
than units.** Concretely, the total absolute error in euros, per product group. If the model still wins
on that, it is a genuine improvement and the segment finding is a curiosity. If it loses, the aggregate
MAE was measuring the wrong thing all along.

A hybrid is often the right answer and rarely proposed: persistence for the top ten, the model for the
tail. There is no rule that one predictor must serve every row, and "which segments does each win on" is
a question worth asking of every model comparison. 05-12 is a whole chapter on error by segment.

## Debugging

### E15

**Benign:** the test period is simply easier. Here, months 17-23 contain members with longer histories, so
the running mean is estimated from more data and predicts better. Nothing is wrong; the two sets are not
interchangeable samples. It is worth confirming rather than assuming - compare the target's spread in
each period.

**A real problem:** the split leaked, or the sets are not what you think they are. The usual cause is a
transformation fitted on all the data before splitting (04-06), so the "test" rows helped compute the
thing being tested. For a baseline specifically, the classic version is a running mean computed with
`expanding()` over an unsorted frame, or with a `shift` in the wrong direction, so the baseline is quietly
using the value it is predicting.

**How to tell:** re-run the baseline on the training period alone with the test rows deleted from the
frame entirely. If the test score changes, information was crossing the boundary.

### E16

**The baseline is using the target.** MAE 0.31 on a target with a standard deviation of 3.56 is not a
per-member mean, it is a copy of the answer.

**The line to look at is the `shift` or the `expanding`.** The two ways this happens:

```
table["mean_so_far"] = table.groupby("member_id").visits.expanding().mean()   # no shift
```

If `visits` at row *t* is included in the mean used to predict `next_visits` at row *t*, that is legal -
this chapter does exactly that, since this month's visits are known at prediction time. But one row
further and it is not: computing the expanding mean of `next_visits` instead of `visits`, or forgetting
to sort by month first, both put the target inside the feature.

**The tell is the number itself.** A baseline that beats every model by a factor of eight is not a good
baseline. It is 04-01's lesson again: *a score better than the problem allows is a bug report.*

## Exam and interview reasoning

### E17

> "Frame it, then compute baselines - before any model. Framing is what one row is, what the target is,
> when the prediction is made and how far ahead. Baselines are the free rules: a constant, last value,
> and the per-entity average, which is the one people forget and usually the hardest to beat. That gives
> me a number the model has to clear, and it takes about twenty minutes. It also usually tells me
> something about the problem - if last-value is very strong the signal is inertia, and if the per-entity
> mean is very strong the variation is between entities rather than within them, which changes what
> features are worth building."

**"What if the baseline wins?"**

> "Then I ship the baseline and say so. That is a successful project, not a failed one - it is one
> `groupby`, it needs no retraining and no serialised model, and I found out in a day instead of a
> quarter. I would also report *why*: here the per-member mean won because members differ a lot from each
> other and little from themselves over time, so there is not much left for a model to learn from the
> columns available. That sentence is the actual deliverable, because it tells whoever asked what kind of
> data would change the answer."

The second half is what separates a good answer from a defensive one. **Being able to recommend against
your own model is the point of computing baselines**, and interviewers are asking whether you would.

## Transfer to a different situation

### E18

Four baselines for daily electricity demand:

1. **Global mean** - demand every day is the annual average. Weak, and the reference everything else is
   measured against.
2. **Persistence** - today equals yesterday. Strong for electricity, since weather and behaviour are
   autocorrelated.
3. **Seasonal persistence: same day last week.** This is the structural one. Demand at 6pm on a Tuesday
   resembles 6pm the previous Tuesday far more than it resembles 6pm on Sunday, because the weekly
   working pattern dominates. Plain persistence gets Saturdays wrong every Monday.
4. **Per-period mean** - the average for this hour, this day of the week, this month, over past years.
   The direct analogue of the per-entity mean that won this chapter, with "entity" being a slot in the
   weekly and annual cycle.

**Hardest to beat: number 3 or 4**, and on hourly data usually a combination - the same hour last week,
adjusted for temperature. Electricity demand is famously dominated by calendar and weather, and a
well-built seasonal baseline is a serious competitor to a full forecasting model. Published forecasting
competitions routinely find that most submissions fail to beat a good seasonal naive baseline.

The transferable point: **the strongest baseline usually encodes the structure everyone in the domain
already knows.** Ask what the obvious pattern is and turn it into a one-line rule.

## Explain it to someone non-technical

### E19

> "Before building anything I check what the simple answers score, so I know what counts as good. It took
> a day and it saved the project: the simplest rule - predict each member's own recent average - turned
> out to beat both models I tried. If I had skipped that step I would have spent three weeks building
> something, reported it as an 18% improvement, and shipped a system that is slightly worse than one line
> of code while costing far more to run. The day told me what the target was. Without it I would have had
> a number with nothing to compare it to."

(88 words.)

## Optional challenge

### E20 - would more data rescue the model?

In [ ]:
fractions = [0.1, 0.25, 0.5, 1.0]
curve = []
for fraction in fractions:
    subset = train.sample(frac=fraction, random_state=0)
    grown = RandomForestRegressor(n_estimators=200, random_state=0, min_samples_leaf=5)
    grown.fit(subset[columns], subset.next_visits)
    curve.append(mean_absolute(grown.predict(test[columns])))

print(pd.DataFrame({"share of training rows": ["%.0f%%" % (100 * f) for f in fractions],
                    "rows": [int(len(train) * f) for f in fractions],
                    "MAE": np.round(curve, 4),
                    "gap to baseline": np.round(np.array(curve) - BEST_BASELINE, 4)}).to_string(index=False))

fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.plot([len(train) * f for f in fractions], curve, "o-", color="#D55E00", label="random forest")
ax.axhline(BEST_BASELINE, color="#0072B2", linewidth=2, label="per-member mean (free)")
ax.set_xlabel("training rows")
ax.set_ylabel("MAE on months 17-23")
ax.set_title("The curve flattens above the baseline, not towards it")
ax.legend()
plt.tight_layout()
plt.show()

**No.** The curve drops sharply from 483 rows to about 1,200 and then flattens between 2.41 and 2.43, never
approaching 2.3640. It even ticks slightly upward from 25% to 100%, which is noise on a single test set -
worth noting rather than explaining, since the differences there are a few thousandths.

**What the curve would need to look like for "collect more data" to be the recommendation:** still
falling steeply at the right-hand edge, with the extrapolated trend crossing the baseline within a
plausible multiple of the data you have. A curve that has flattened is telling you the model has
extracted what these columns contain; more rows of the same columns will not add anything, and the
remaining gap is not a data-volume problem.

Two honest footnotes.

The 100% point here reads 2.4288 while the chapter's identical forest read 2.4353. Nothing changed except
that `sample(frac=1.0)` returns the same rows in a different order, and the forest's bootstrap draws
depend on that order. **A 0.0065 wobble from row order alone** is a useful calibration for how seriously
to take the fourth decimal place anywhere in this chapter - and a reminder that a learning curve from a
single fit per point should have error bars on it.

And the diagnosis "flat curve means more data will not help" is specifically about **more rows of the
same columns**. It says nothing about better columns. The right follow-up here is not more members, it is
a feature that captures something the four available columns do not - a cancellation notice, a change of
address, a price rise - which is a data-collection question rather than a modelling one, and is usually
the honest answer when a model cannot beat its baseline.